# Exploratory Data Analysis — Prediction Market Forecasting

This notebook explores the data pipelines built in Section 3:
1. **Polymarket** — market discovery + price history
2. **Kalshi** — market discovery + candlestick data
3. **Exogenous** — Google Trends, FRED macro data
4. **Preprocessing** — filtering, returns, walk-forward splits

In [ ]:
import sys, json, asyncio, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")

logging.basicConfig(level=logging.INFO, format="%(name)s %(levelname)s %(message)s")

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 120})

print("Setup OK")

## 1. Polymarket — Market Discovery & Price History

In [ ]:
import aiohttp
from src.data.polymarket import PolymarketClient

GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE = "https://clob.polymarket.com"

async def discover_polymarket(n_pages=20):
    """Fetch active Polymarket markets (multiple pages to ensure we capture high-volume ones)."""
    all_markets = []
    async with aiohttp.ClientSession() as session:
        for page in range(n_pages):
            params = {"limit": 100, "offset": page * 100, "active": "true", "closed": "false"}
            async with session.get(f"{GAMMA_BASE}/markets", params=params) as resp:
                batch = await resp.json()
            if not batch:
                break
            all_markets.extend(batch)
    
    df = pd.DataFrame(all_markets)
    df["volumeNum"] = pd.to_numeric(df["volumeNum"], errors="coerce").fillna(0)
    df["liquidityNum"] = pd.to_numeric(df["liquidityNum"], errors="coerce").fillna(0)
    if "category" not in df.columns:
        df["category"] = None
    
    # Sort by volume descending — API doesn't support numeric sorting
    df = df.sort_values("volumeNum", ascending=False).reset_index(drop=True)
    return df

poly_markets = await discover_polymarket(n_pages=20)
print(f"Discovered {len(poly_markets)} active Polymarket markets")
print(f"Top 15 by volume:")
poly_markets[["question", "volumeNum", "liquidityNum"]].head(15)

### Volume distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume distribution (log scale)
axes[0].hist(np.log10(poly_markets["volumeNum"].clip(lower=1)), bins=40, edgecolor="k", alpha=0.7)
axes[0].set_xlabel("log10(Volume USD)")
axes[0].set_ylabel("Count")
axes[0].set_title("Polymarket — Volume Distribution (Active Markets)")
axes[0].axvline(np.log10(100_000), color="r", ls="--", label="$100k threshold")
axes[0].axvline(np.log10(1_000_000), color="orange", ls="--", label="$1M threshold")
axes[0].legend()

# Category breakdown (may be all None for active markets)
cat_vals = poly_markets["category"].fillna("Uncategorized")
cat_counts = cat_vals.value_counts().head(15)
cat_counts.plot.barh(ax=axes[1], edgecolor="k", alpha=0.7)
axes[1].set_xlabel("Number of Markets")
axes[1].set_title("Polymarket — Top Categories")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Stats
above_100k = (poly_markets["volumeNum"] >= 100_000).sum()
above_1m = (poly_markets["volumeNum"] >= 1_000_000).sum()
print(f"Markets with volume >= $100k: {above_100k}")
print(f"Markets with volume >= $1M: {above_1m}")
print(f"Median volume: ${poly_markets['volumeNum'].median():,.0f}")
print(f"Mean volume: ${poly_markets['volumeNum'].mean():,.0f}")

### Price history for top markets

In [ ]:
# Fetch price history for the most liquid markets (top 10 by volume)
top_markets = poly_markets.head(10)
print(f"Fetching price history for top {len(top_markets)} markets by volume...")

async def fetch_top_histories(markets_df):
    """Fetch price histories for the most liquid markets."""
    histories = {}
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=30)) as session:
        for _, row in markets_df.iterrows():
            raw_ids = row.get("clobTokenIds", "[]")
            raw_out = row.get("outcomes", "[]")
            token_ids = json.loads(raw_ids) if isinstance(raw_ids, str) else (raw_ids or [])
            outcomes = json.loads(raw_out) if isinstance(raw_out, str) else (raw_out or [])
            question = str(row.get("question", ""))[:50]
            volume = row.get("volumeNum", 0)
            
            if not token_ids:
                continue
            
            # Fetch Yes side only
            params = {"market": token_ids[0], "interval": "max", "fidelity": 60}
            try:
                async with session.get(f"{CLOB_BASE}/prices-history", params=params) as resp:
                    data = await resp.json()
                    history = data.get("history", [])
            except Exception as e:
                print(f"  Error fetching {question}: {e}")
                continue
            
            if history:
                df = pd.DataFrame(history)
                df.columns = ["timestamp", "price"]
                df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
                df = df.set_index("timestamp").sort_index()
                histories[question] = df
                print(f"  {question}: {len(df)} pts, vol=${volume:,.0f}")
            else:
                print(f"  {question}: no price data (vol=${volume:,.0f})")
    return histories

histories = await fetch_top_histories(top_markets)
print(f"\nFetched price history for {len(histories)} / {len(top_markets)} markets")

In [ ]:
# Plot price trajectories
if not histories:
    print("No price histories available to plot.")
else:
    n_plots = min(len(histories), 6)
    n_cols = min(n_plots, 3)
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), squeeze=False)
    axes = axes.flatten()

    for i, (question, df) in enumerate(list(histories.items())[:n_plots]):
        ax = axes[i]
        ax.plot(df.index, df["price"], linewidth=0.8)
        ax.set_title(question[:40] + "...", fontsize=9)
        ax.set_ylabel("Price (probability)")
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, alpha=0.3)

    for j in range(n_plots, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Polymarket — Sample Contract Price Trajectories", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

### Price distribution & return statistics

In [ ]:
# Compute returns for each contract and show distributions
if not histories:
    print("No data to analyze.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    all_prices = []
    all_returns = []

    for question, df in histories.items():
        all_prices.extend(df["price"].values)
        returns = df["price"].pct_change().dropna()
        all_returns.extend(returns.values)

    # Price distribution
    axes[0].hist(all_prices, bins=50, edgecolor="k", alpha=0.7)
    axes[0].set_xlabel("Price (probability)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Price Distribution (all contracts)")

    # Return distribution
    returns_arr = np.array(all_returns)
    returns_clipped = returns_arr[(returns_arr > -1) & (returns_arr < 1) & np.isfinite(returns_arr)]

    if len(returns_clipped) > 0:
        axes[1].hist(returns_clipped, bins=80, edgecolor="k", alpha=0.7)
        axes[1].set_xlabel("Simple Return")
        axes[1].set_title("Return Distribution")
        axes[1].axvline(0, color="r", ls="--", alpha=0.5)

        # QQ plot of returns
        from scipy import stats
        stats.probplot(returns_clipped, dist="norm", plot=axes[2])
        axes[2].set_title("QQ Plot of Returns vs Normal")

        plt.tight_layout()
        plt.show()

        print(f"Return statistics (across all contracts):")
        print(f"  Mean:     {np.nanmean(returns_clipped):.6f}")
        print(f"  Std:      {np.nanstd(returns_clipped):.6f}")
        print(f"  Skewness: {stats.skew(returns_clipped):.4f}")
        print(f"  Kurtosis: {stats.kurtosis(returns_clipped):.4f}")
    else:
        print("Not enough return data to analyze.")
        plt.close(fig)

## 2. Kalshi — Market Discovery

In [ ]:
import requests
import time

KALSHI_BASE = "https://api.elections.kalshi.com/trade-api/v2"

# Discover Kalshi events first, then get their markets
# The default /markets pagination misses high-volume markets
events = []
cursor = None
for _ in range(5):
    params = {"limit": 200}
    if cursor:
        params["cursor"] = cursor
    resp = requests.get(f"{KALSHI_BASE}/events", params=params, timeout=30)
    data = resp.json()
    batch = data.get("events", [])
    events.extend(batch)
    cursor = data.get("cursor")
    if not cursor or not batch:
        break

print(f"Discovered {len(events)} events")

# Fetch markets for top events and compute total volume per event
event_stats = []
kalshi_all_markets = []
for e in events[:100]:  # top 100 events
    et = e.get("event_ticker", "")
    try:
        resp = requests.get(f"{KALSHI_BASE}/markets", params={"limit": 100, "event_ticker": et}, timeout=10)
        mkts = resp.json().get("markets", [])
        total_vol = sum(float(m.get("volume_fp", 0)) for m in mkts)
        event_stats.append({
            "title": e.get("title", "")[:55],
            "category": e.get("category", ""),
            "n_markets": len(mkts),
            "total_volume_contracts": total_vol,
        })
        for m in mkts:
            m["_event_title"] = e.get("title", "")
            m["_category"] = e.get("category", "")
        kalshi_all_markets.extend(mkts)
    except Exception:
        pass
    time.sleep(0.06)

event_stats.sort(key=lambda x: x["total_volume_contracts"], reverse=True)
event_df = pd.DataFrame(event_stats)

print(f"\nTop 15 Kalshi events by volume (contracts traded):")
display(event_df.head(15))

# Build per-market DataFrame
kalshi_df = pd.DataFrame(kalshi_all_markets)
kalshi_df["volume"] = pd.to_numeric(kalshi_df.get("volume_fp", 0), errors="coerce").fillna(0)
kalshi_df["category"] = kalshi_df.get("_category", "")
kalshi_df = kalshi_df.sort_values("volume", ascending=False).reset_index(drop=True)

print(f"\nKalshi markets sampled: {len(kalshi_df)}")
print(f"NOTE: Kalshi volume_fp is in contracts, not USD.")
print(f"      Each contract settles at $0 or $1, so notional ~ volume * avg_price")
print(f"Markets with volume > 0: {(kalshi_df['volume'] > 0).sum()}")
print(f"Markets with volume >= 10k contracts: {(kalshi_df['volume'] >= 10_000).sum()}")
print(f"Markets with volume >= 100k contracts: {(kalshi_df['volume'] >= 100_000).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume distribution
vol_nonzero = kalshi_df.loc[kalshi_df["volume"] > 0, "volume"]
if len(vol_nonzero) > 0:
    axes[0].hist(np.log10(vol_nonzero.clip(lower=1)), bins=40, edgecolor="k", alpha=0.7, color="coral")
    axes[0].axvline(np.log10(10_000), color="r", ls="--", label="10k contracts")
    axes[0].axvline(np.log10(100_000), color="orange", ls="--", label="100k contracts")
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "No markets with volume > 0", ha="center", va="center", transform=axes[0].transAxes)
axes[0].set_xlabel("log10(Volume, contracts)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Kalshi — Volume Distribution (n={len(vol_nonzero)} with vol > 0)")

# Category breakdown from events
if "category" in kalshi_df.columns:
    cat_counts = kalshi_df["category"].replace("", "Unknown").value_counts().head(10)
    cat_counts.plot.barh(ax=axes[1], edgecolor="k", alpha=0.7, color="coral")
    axes[1].set_xlabel("Number of Markets")
    axes[1].set_title("Kalshi — Market Categories (from events)")
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 3. Exogenous Data — Google Trends & FRED

In [ ]:
from src.data.exogenous import fetch_google_trends

try:
    trends = fetch_google_trends(
        keywords=["Polymarket", "Kalshi", "prediction market", "election betting"],
        timeframe="2023-01-01 2025-12-31",
    )
except Exception as e:
    print(f"Google Trends fetch failed: {e}")
    trends = pd.DataFrame()

if not trends.empty:
    fig, ax = plt.subplots(figsize=(14, 5))
    trends.plot(ax=ax, linewidth=1.5)
    ax.set_ylabel("Google Trends Index (0-100)")
    ax.set_title("Google Trends — Prediction Market Keywords")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"\nTrends shape: {trends.shape}")
    print(f"Date range: {trends.index.min()} — {trends.index.max()}")
    display(trends.describe())
else:
    print("No Google Trends data available.")

In [ ]:
# FRED macro data — requires FRED_API_KEY
# Set your key: import os; os.environ["FRED_API_KEY"] = "your_key_here"
# Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html

import os
from src.data.exogenous import fetch_all_fred

if os.environ.get("FRED_API_KEY"):
    fred = fetch_all_fred(start="2023-01-01", end="2025-12-31")
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()
    
    for i, col in enumerate(fred.columns[:6]):
        fred[col].dropna().plot(ax=axes[i], linewidth=1)
        axes[i].set_title(col)
        axes[i].grid(True, alpha=0.3)
    
    plt.suptitle("FRED Macro Indicators", fontsize=13)
    plt.tight_layout()
    plt.show()
    
    fred.describe()
else:
    print("FRED_API_KEY not set. Skipping FRED data.") 
    print("Set it with: os.environ['FRED_API_KEY'] = 'your_key'")
    print("Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html")

## 4. Preprocessing Pipeline Test

In [ ]:
from src.data.preprocessing import (
    compute_returns, winsorize_returns,
    temporal_split, walk_forward_splits
)

if not histories:
    print("No price histories to preprocess.")
else:
    # Use the first contract with enough data
    sample_name = list(histories.keys())[0]
    sample = histories[sample_name].copy()
    print(f"Sample contract: '{sample_name}'")
    print(f"Raw: {len(sample)} hourly points, {sample.index.min()} — {sample.index.max()}")

    # Resample to daily
    daily = sample[["price"]].resample("1D").last().ffill().dropna()
    print(f"Daily: {len(daily)} rows")

    # Compute returns
    daily = compute_returns(daily, price_col="price")
    daily = winsorize_returns(daily, return_col="log_return", limits=(0.01, 0.01))
    print(f"Columns after preprocessing: {list(daily.columns)}")

    daily.head()

In [ ]:
# Visualize preprocessing results
if histories and "daily" in dir():
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    # Price series
    axes[0, 0].plot(daily.index, daily["price"], linewidth=1)
    axes[0, 0].set_title(f"Daily Price — {sample_name[:40]}")
    axes[0, 0].set_ylabel("Price")
    axes[0, 0].grid(True, alpha=0.3)

    # Returns
    axes[0, 1].bar(daily.index, daily["log_return"], width=1, alpha=0.7)
    axes[0, 1].set_title("Daily Log Returns (winsorized)")
    axes[0, 1].set_ylabel("Log Return")
    axes[0, 1].axhline(0, color="k", linewidth=0.5)
    axes[0, 1].grid(True, alpha=0.3)

    # Return distribution
    log_rets = daily["log_return"].dropna()
    if len(log_rets) > 2:
        axes[1, 0].hist(log_rets, bins=30, edgecolor="k", alpha=0.7)
        axes[1, 0].set_xlabel("Log Return")
        axes[1, 0].set_title("Return Distribution")
        axes[1, 0].axvline(0, color="r", ls="--", alpha=0.5)

        # Autocorrelation
        from pandas.plotting import autocorrelation_plot
        autocorrelation_plot(log_rets, ax=axes[1, 1])
        axes[1, 1].set_title("Return Autocorrelation")
        axes[1, 1].set_xlim(0, min(30, len(log_rets) - 1))

    plt.tight_layout()
    plt.show()
else:
    print("No preprocessed data to visualize.")

## 5. Cross-Platform Comparison

How do Polymarket and Kalshi compare in terms of market coverage and volume?

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Markets sampled",
        "Markets with volume > 0",
        "Volume unit",
        "Median volume (non-zero)",
        "Mean volume (non-zero)",
        "Max volume",
    ],
    "Polymarket": [
        len(poly_markets),
        (poly_markets["volumeNum"] > 0).sum(),
        "USD",
        f"${poly_markets.loc[poly_markets['volumeNum'] > 0, 'volumeNum'].median():,.0f}",
        f"${poly_markets.loc[poly_markets['volumeNum'] > 0, 'volumeNum'].mean():,.0f}",
        f"${poly_markets['volumeNum'].max():,.0f}",
    ],
    "Kalshi": [
        len(kalshi_df),
        (kalshi_df["volume"] > 0).sum(),
        "Contracts (each settles $0-$1)",
        f"{kalshi_df.loc[kalshi_df['volume'] > 0, 'volume'].median():,.0f}",
        f"{kalshi_df.loc[kalshi_df['volume'] > 0, 'volume'].mean():,.0f}",
        f"{kalshi_df['volume'].max():,.0f}",
    ],
})

comparison.set_index("Metric")

## 6. Key Takeaways & Next Steps

**Observations:**
- **Kalshi volume is in contracts, not USD** — direct comparison with Polymarket (which reports USD volume) is misleading. Kalshi's top events (e.g. Fed Chair nomination: ~217M contracts) represent significant real volume.
- **Kalshi's high-volume markets are best discovered via the events endpoint**, not the default `/markets` pagination (which returns in creation order, burying liquid markets).
- Google Trends data shows clear spikes around major events — promising exogenous signal.
- Return distributions are heavy-tailed (high kurtosis), confirming the need for winsorization.
- Categories on Polymarket are not returned for active markets — may need to infer from question text. Kalshi has structured categories via the events API.

**Next steps:**
1. Feature engineering (Section 4): endogenous + exogenous features
2. Normalize volume units for cross-platform comparison (contracts * avg_price for Kalshi)
3. Update `kalshi.py` ingestion to use events-based discovery for better coverage
4. Build the unified feature matrix